# Colab 02b — nnU-Net (pipeline propio, enmienda E3)
5 folds ↔ 5 semillas, configuración 2d, 50 épocas. **Este pipeline predice y evalúa el test** (80 pos + 40 neg), por eso copia el test: ejecútalo **después** de congelar `protocolo.md` (lo verifica la celda de congelamiento).
Recomendado: GPU **L4**. Tiempo ≈ 1–2 h por fold. Reanudable.

In [ ]:
# Celda 1 — GPU
import os
!nvidia-smi | head -15

In [ ]:
# Celda 2 — dependencias
%pip install -q segmentation-models-pytorch monai einops timm scipy scikit-image pandas opencv-python-headless tqdm pytest matplotlib onnx onnxruntime nnunetv2 nibabel
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
assert torch.cuda.is_available(), "Sin GPU: activa una GPU (Runtime → Change runtime type)"

In [ ]:
# Celda 3 — montar Drive y fijar rutas
from google.colab import drive
drive.mount('/content/drive')
CARPETA_DRIVE = "/content/drive/MyDrive/proyecto_tb"      # AJUSTAR si cambia
RAIZ_LOCAL = "/content/proyecto_tb"                        # copia local (rápida)
RES_HPO = f"{CARPETA_DRIVE}/resultados"                    # donde quedó la búsqueda (hpo/)
RES_E3 = f"{CARPETA_DRIVE}/resultados_e3"                  # resultados finales del Entregable 3
HPO_CFG = f"{RES_HPO}/hpo/mejor_config.json"
SPL = f"{RAIZ_LOCAL}/datos/splits"
os.makedirs(RES_E3, exist_ok=True)
assert os.path.isdir(f"{CARPETA_DRIVE}/codigo"), "Falta codigo/ en Drive"
assert os.path.isdir(f"{CARPETA_DRIVE}/datos/splits"), "Falta datos/splits"
print("Drive OK:", CARPETA_DRIVE, "\nResultados E3:", RES_E3)

In [ ]:
# Celda 4 — copiar código y datos a disco local: train/val + TEST + GT (el test se lee solo aquí)
import csv, shutil
def a_local(r, raiz):
    n = r.replace("\\", "/"); i = n.find("datos/"); assert i >= 0, r
    return os.path.join(raiz, n[i:])
shutil.copytree(f"{CARPETA_DRIVE}/codigo", f"{RAIZ_LOCAL}/codigo", dirs_exist_ok=True)
os.makedirs(SPL, exist_ok=True)
for nombre in ("train", "val", "test_pos", "test_neg", "test_pos_reserva"):
    p = f"{CARPETA_DRIVE}/datos/splits/{nombre}.csv"
    if not os.path.exists(p): print("  (no existe", nombre, ")"); continue
    filas = list(csv.DictReader(open(p, newline="", encoding="utf-8")))
    campos = list(filas[0].keys())
    for f in filas:
        dst, src = a_local(f["ruta"], RAIZ_LOCAL), a_local(f["ruta"], CARPETA_DRIVE)
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(src, dst)
        f["ruta"] = dst
    with open(f"{SPL}/{nombre}.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=campos)
        w.writeheader(); w.writerows(filas)
    print(f"  {nombre:16s} {len(filas)} filas")
for mf in ("split_manifest_v2.json", "split_manifest.json"):
    if os.path.exists(f"{CARPETA_DRIVE}/datos/splits/{mf}"):
        shutil.copy2(f"{CARPETA_DRIVE}/datos/splits/{mf}", f"{SPL}/{mf}")

gt = f"{CARPETA_DRIVE}/datos/ground_truth_test"
assert os.path.isdir(gt), "Falta datos/ground_truth_test en Drive (GT plata)"
shutil.copytree(gt, f"{RAIZ_LOCAL}/datos/ground_truth_test", dirs_exist_ok=True)
print("GT plata copiado")
import sys; sys.path.insert(0, f"{RAIZ_LOCAL}/codigo")

In [ ]:
# Celda — congelar / verificar protocolo.md (SHA-256)
import hashlib
p = f"{CARPETA_DRIVE}/docs/protocolo.md"
assert os.path.exists(p), "Falta docs/protocolo.md en Drive"
h = hashlib.sha256(open(p, "rb").read()).hexdigest()
f = f"{RES_E3}/protocolo.sha256"
if os.path.exists(f):
    assert open(f).read().split()[0] == h, "protocolo.md CAMBIÓ después de congelarse: registra una enmienda"
    print("protocolo verificado:", h[:16], "…")
else:
    open(f, "w").write(h + "  protocolo.md\n"); print("protocolo CONGELADO:", h)

In [ ]:
# Celda 5 — prueba de humo: conversión + planificación (sin entrenar)
!python "{RAIZ_LOCAL}/codigo/entrenamiento_nnunet.py" --solo-preparar --resultados "{RES_E3}" --splits-dir "{SPL}"

In [ ]:
# Celda 6 — ENTRENAR los 5 folds (reanudable: si se corta, repite la celda)
!python "{RAIZ_LOCAL}/codigo/entrenamiento_nnunet.py" --resultados "{RES_E3}" --splits-dir "{SPL}" --red 2d --epocas 50

In [ ]:
# Celda 7 — estado de los folds
import json, glob
for p in sorted(glob.glob(f"{RES_E3}/nnunet/seed_*/resultado.json")):
    r = json.load(open(p)); print(p.split("/")[-2], r.get("estado"), r.get("dice_test"))